# SRQ-FLY D1 — 20-task ImageNet-R train-only drift study
Run every cell in order on a Colab T4 GPU. D1 reuses the verified train/WTA caches created by D0, never opens `test.pt`, and prints paired exact/SRQ drift after every task. Return the final ZIP for audit.

In [ ]:
# === Edit repository/Drive paths only. Do not edit seed, config, or gates. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/srq-fly-d1'
WORK_DIR = '/content/SOHO-CL'
DRIVE_ROOT = '/content/drive/MyDrive/T-SOHO'
DRIVE_TRAIN_CACHE = f'{DRIVE_ROOT}/imagenetr_train_feature_cache_seed2025'
TRAIN_CACHE_DIR = '/content/imagenetr_train_feature_cache_seed2025'
DRIVE_LARGE_WTA_CACHE = f'{DRIVE_ROOT}/tail_fly_imagenetr_wta_cache_seed2025'
LARGE_WTA_CACHE_DIR = '/content/srq_fly_wta_h10000_seed2025'
DRIVE_COMPACT_WTA_CACHE = f'{DRIVE_ROOT}/srq_fly_wta_h4096_seed2025'
COMPACT_WTA_CACHE_DIR = '/content/srq_fly_wta_h4096_seed2025'
OUTPUT_DIR = f'{DRIVE_ROOT}/srq_fly_imagenetr_d1_seed2025'
SEED = 2025
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CONFIG_SHA256 = 'f61da98c3d59d687ce10a4f9ecd5b2ec56251ad4d712130e55d33577a2685dde'

In [ ]:
# Runtime setup. The initial chdir prevents Colab's deleted-working-directory failure.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
os.chdir('/content')
repo_path = Path(WORK_DIR)
if repo_path.exists(): shutil.rmtree(repo_path)
clone = subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_GIT_URL, WORK_DIR], text=True, capture_output=True)
print(clone.stdout, clone.stderr, sep='')
assert clone.returncode == 0, f'Clone failed ({clone.returncode}). Confirm feature/srq-fly-d1 was pushed.'
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
config_path = Path('configs/srq_fly_imagenetr_d1_train_only.json')
assert hashlib.sha256(config_path.read_bytes()).hexdigest() == CONFIG_SHA256, 'Locked config identity mismatch.'
print('repo commit:', commit)
print('GPU:', torch.cuda.get_device_name(0))
print('locked seed:', SEED, '| config SHA-256:', CONFIG_SHA256)

In [ ]:
# Restore D0's verified experiment caches with one progress line per file.
def restore_required_cache(drive_path, local_path, required):
    source, target = Path(drive_path), Path(local_path)
    missing_source = [name for name in required if not (source/name).is_file()]
    assert not missing_source, f'Missing D0 Drive cache files under {source}: {missing_source}'
    target.mkdir(parents=True, exist_ok=True)
    for name in required:
        item = source/name
        if not (target/name).is_file():
            print('COPY', item, f'{item.stat().st_size/2**20:.1f} MiB', flush=True)
            shutil.copy2(item, target/name)
    print('CACHE READY:', target, flush=True)
restore_required_cache(DRIVE_TRAIN_CACHE, TRAIN_CACHE_DIR, ['metadata.json', 'train.pt'])
restore_required_cache(DRIVE_LARGE_WTA_CACHE, LARGE_WTA_CACHE_DIR, ['metadata.json', 'projection.pt', 'train_codes.pt'])
restore_required_cache(DRIVE_COMPACT_WTA_CACHE, COMPACT_WTA_CACHE_DIR, ['metadata.json', 'projection.pt', 'train_codes.pt'])
metadata = json.loads(Path(TRAIN_CACHE_DIR, 'metadata.json').read_text())
assert metadata['dataset'] == 'ImageNet-R' and metadata['checkpoint_sha256'] == CHECKPOINT_SHA256
assert metadata['feature_dim'] == 768 and metadata['finite'] is True
assert metadata['test_features_materialized'] is False
assert not Path(TRAIN_CACHE_DIR, 'test.pt').exists(), 'Held-out test cache must remain absent.'
print('D0 cache preflight: PASS', metadata['train_shape'], '| test.pt absent')

In [ ]:
# Q0/D1 correctness gate: synthetic data only.
tests = ['tests/test_srq_fly_math.py', 'tests/test_srq_fly_learner.py', 'tests/test_srq_fly_d0.py', 'tests/test_srq_fly_d1.py']
subprocess.run([sys.executable, '-m', 'pytest', '-q', *tests], check=True)
print('SRQ-FLY D1 correctness gate: PASS')

In [ ]:
# Locked 20-task run. START/DONE mark methods; each TASK line reports completed progress.
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)
shutil.copy2(config_path, output_path/'locked_config.json')
command = [sys.executable, '-u', 'tools/srq_fly_d1.py', '--config', str(config_path), '--feature-cache-dir', TRAIN_CACHE_DIR, '--large-code-cache-dir', LARGE_WTA_CACHE_DIR, '--compact-code-cache-dir', COMPACT_WTA_CACHE_DIR, '--output-dir', OUTPUT_DIR, '--device', 'cuda', '--require-test-hidden']
print('Starting SRQ-FLY D1: 6 methods over all 20 train-validation tasks.', flush=True)
print('Paired TASK lines show exact AA, SRQ AA, agreement, and logit error.', flush=True)
print('Completed method units resume automatically from Drive.', flush=True)
started = time.time()
completed = subprocess.run(command)
print(f'Runner elapsed: {(time.time()-started)/60:.1f} minutes', flush=True)
assert completed.returncode == 0, 'D1 failed; send the complete traceback without editing config.'
assert (output_path/'d1_results.json').is_file()
assert not Path(TRAIN_CACHE_DIR, 'test.pt').exists()
print('SRQ-FLY D1 process: COMPLETE')

In [ ]:
# Summarize, download evidence, then STOP.
import pandas as pd
result = json.loads((output_path/'d1_results.json').read_text())
rows = [{'method':x['method'], 'status':x['status'], 'validation_AA':x.get('validation_average_accuracy'), 'final_seen_validation_accuracy':x.get('stage_accuracy', [None])[-1], 'persistent_state_bytes':x.get('persistent_state_bytes'), 'max_solver_residual':x.get('maximum_solver_relative_residual')} for x in result['results']]
display(pd.DataFrame(rows).sort_values('validation_AA', ascending=False))
paired = pd.DataFrame(result['paired_diagnostics'])
display(paired[['task','exact_accuracy','approximate_accuracy','accuracy_gap_pp','prediction_agreement','relative_logit_frobenius_error']])
print('decision:', result['status'])
print('gates:', json.dumps(result['gates'], indent=2))
archive = shutil.make_archive('/content/srq_fly_imagenetr_d1_train_only', 'zip', root_dir=OUTPUT_DIR)
print('artifact SHA-256:', hashlib.sha256(Path(archive).read_bytes()).hexdigest())
from google.colab import files
files.download(archive)
print('STOP. Send the ZIP for audit; do not evaluate ImageNet-R test.')